# LoRA এবং QLoRA

চারটি demo:

1. Raw PyTorch-এ স্ক্র্যাচ থেকে তৈরি একটি `LoRALinear` module: একটি frozen `nn.Linear` সাথে একটি trainable low-rank A/B update।
2. এটিকে ছোট একটি toy regression task-এ train করা এবং **সরাসরি** যাচাই করা যে frozen base weight একটি bit-ও বদলায় না, অথচ A ও B বদলায়।
3. Parameter-count তুলনা: বাস্তবসম্মত আকারের একটি weight matrix-এর জন্য dense fine-tuning বনাম কয়েকটি rank-এ LoRA, যা বাস্তব PyTorch parameter count-এর সাথে মিলিয়ে দেখা হয় (শুধু formula নয়)।
4. Frozen base-এর QLoRA-ধাঁচের 4-bit block quantization-এর একটি স্ক্র্যাচ থেকে তৈরি simulation: quantization error এবং memory footprint, fp32/fp16 storage-এর তুলনায়।

কোনো বাইরের PEFT library ব্যবহার করা হয়নি বা দরকারও নেই — এই lesson mechanism-টি নিজেই তৈরি করে। বাস্তব workflow-এ Hugging Face `peft` (LoraConfig / get_peft_model) ব্যবহৃত হয়, যা Lesson 5-এ আলোচিত।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। (Demo 1 ও 2 একই cell-এ একসাথে চলে — `lora_layer_demo()`।)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1-2. LoRALinear: frozen `nn.Linear` সাথে trainable low-rank update — train করা ও যাচাই

`LoRALinear` একটি frozen base layer-কে মোড়ায় এবং `W' = W + (alpha/r) * B @ A` update যোগ করে; `B` zero দিয়ে initialize হয় বলে শুরুতে module-টি হুবহু base-এর মতো আচরণ করে। Demo-টি একটি toy regression task-এ শুধু A ও B train করে, তারপর সরাসরি পরীক্ষা করে যে frozen `base.weight`/`base.bias` একটুও বদলায়নি, loss কমেছে (কিন্তু low-rank সীমাবদ্ধতার কারণে zero-তে পৌঁছায় না), এবং merged weight দিয়ে forward pass হুবহু একই output দেয় — অর্থাৎ deployment-এ zero extra latency।

In [ ]:
# ---------------------------------------------------------------------------
# 1. LoRALinear: একটি frozen nn.Linear সাথে একটি trainable low-rank update
# ---------------------------------------------------------------------------

class LoRALinear(nn.Module):
    """একটি frozen nn.Linear(in_features, out_features)-কে একটি trainable
    low-rank update W' = W + (alpha/r) * B @ A দিয়ে মোড়ায়, হুবহু README-এর মতো।"""

    def __init__(self, in_features, out_features, r, alpha=None, bias=True):
        super().__init__()
        self.r = r
        self.alpha = alpha if alpha is not None else r  # একটি প্রচলিত default: alpha == r
        self.scaling = self.alpha / self.r

        # মূল pretrained layer -- frozen। বাস্তব workflow-এ এতে আগে থেকেই
        # pretrained weights থাকত; এখানে এটি স্বাভাবিকভাবে initialize করা হয়েছে
        # শুধু "কোনো একটি pretrained weight matrix"-এর stand-in হিসেবে।
        self.base = nn.Linear(in_features, out_features, bias=bias)
        for p in self.base.parameters():
            p.requires_grad_(False)

        # A: r x in_features, ছোট random init (symmetry ভাঙে)।
        # B: out_features x r, ZERO দিয়ে initialize করা, যাতে step 0-তে
        #    B @ A == 0 হয় এবং এই module-টি সংখ্যাগতভাবে `base`-এর হুবহু সমান হয়।
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))

    def forward(self, x):
        base_out = self.base(x)
        lora_out = (x @ self.A.T) @ self.B.T
        return base_out + self.scaling * lora_out

    def merged_weight(self):
        """W + (alpha/r) * B @ A -- zero-extra-latency inference-এর জন্য যা deploy করবেন।"""
        return self.base.weight + self.scaling * (self.B @ self.A)


def lora_layer_demo():
    print("=" * 78)
    print("1-2. A FROZEN BASE + TRAINABLE LOW-RANK UPDATE, TRAINED AND VERIFIED")
    print("=" * 78)

    in_features, out_features, r = 32, 32, 4
    layer = LoRALinear(in_features, out_features, r=r)

    base_weight_before = layer.base.weight.detach().clone()
    base_bias_before = layer.base.bias.detach().clone()

    # একটি toy task: একটি নির্দিষ্ট random linear map শেখা, যা randomly-initialized
    # frozen base আগে থেকে implement করে না। কেবল A/B-ই এটি fit করতে পারে,
    # কারণ base.weight/bias-এর requires_grad=False।
    target_map = torch.randn(out_features, in_features)
    x_train = torch.randn(256, in_features)
    y_train = x_train @ target_map.T

    trainable_params = [p for p in layer.parameters() if p.requires_grad]
    print(f"Trainable parameters: {sum(p.numel() for p in trainable_params)} "
          f"(A: {layer.A.numel()}, B: {layer.B.numel()})")
    print(f"Frozen parameters:    {sum(p.numel() for p in layer.parameters() if not p.requires_grad)} "
          f"(base.weight + base.bias)")

    optimizer = torch.optim.Adam(trainable_params, lr=1e-2)
    initial_loss = None
    for step in range(1, 301):
        pred = layer(x_train)
        loss = F.mse_loss(pred, y_train)
        if initial_loss is None:
            initial_loss = loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 100 == 0 or step == 1:
            print(f"  step {step:4d}  loss = {loss.item():.5f}")

    final_loss = loss.item()

    # মূল PEFT দাবিটি, ধরে না নিয়ে সরাসরি যাচাই করা হচ্ছে:
    weight_unchanged = torch.equal(layer.base.weight, base_weight_before)
    bias_unchanged = torch.equal(layer.base.bias, base_bias_before)
    weight_grad_is_none = layer.base.weight.grad is None
    A_changed = not torch.allclose(layer.A, torch.zeros_like(layer.A) + layer.A[0, 0])  # sanity: A degenerate নয়
    B_is_nonzero = layer.B.abs().sum().item() > 0

    print(f"\nFrozen base.weight identical to its pre-training value: {weight_unchanged}")
    print(f"Frozen base.bias identical to its pre-training value:   {bias_unchanged}")
    print(f"Frozen base.weight.grad is None (never touched by backward's update step): {weight_grad_is_none}")
    print(f"B moved away from its zero initialization (sum|B| = {layer.B.abs().sum().item():.4f} > 0): {B_is_nonzero}")
    print(f"\nLoss fell from {initial_loss:.5f} to {final_loss:.5f} training ONLY A and B --")
    print("the frozen base contributed a fixed, unhelpful random transform throughout,")
    print(f"and the rank-{r} update alone did all the learning. It plateaus above zero")
    print(f"because the target map needs a full-rank ({in_features}-rank) correction and a")
    print(f"rank-{r} B@A update is mathematically incapable of reproducing all of it --")
    print("a direct, hands-on illustration of what 'low-rank' actually constrains.")

    # Merging যাচাই: merged weight-এর উচিত trained forward pass-টি হুবহু পুনরুৎপাদন করা।
    merged_w = layer.merged_weight()
    with torch.no_grad():
        out_unmerged = layer(x_train[:8])
        out_merged = x_train[:8] @ merged_w.T + layer.base.bias
    max_diff = (out_unmerged - out_merged).abs().max().item()
    print(f"\nMax difference between LoRA forward pass and the merged-weight forward pass: "
          f"{max_diff:.2e}")
    print("-> Merging A/B into W once, offline, reproduces training-time behavior exactly --")
    print("   this is why LoRA adds zero extra latency once merged for deployment.")


lora_layer_demo()

## 3. Parameter count: dense fine-tuning বনাম কয়েকটি rank-এ LoRA

একটি 4096 x 4096 weight matrix (7B-শ্রেণির model-এর মতো) নিয়ে rank `r` = 1, 4, 16, 64-এর জন্য LoRA-র trainable parameter সংখ্যা `r*(d+k)` formula দিয়ে হিসাব করা হয়, এবং প্রতিটি ক্ষেত্রে একটি বাস্তব `LoRALinear` instantiate করে তার আসল parameter count-এর সাথে মিলিয়ে দেখা হয়। r=64-এও LoRA dense-এর মাত্র ~3% parameter train করে।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Parameter count: dense fine-tuning বনাম কয়েকটি rank-এ LoRA
# ---------------------------------------------------------------------------

def parameter_count_demo():
    print("\n" + "=" * 78)
    print("3. TRAINABLE PARAMETERS: FULL FINE-TUNING vs. LoRA AT SEVERAL RANKS")
    print("=" * 78)

    d = k = 4096  # একটি বাস্তবসম্মত attention/FFN-projection matrix size (7B-model শ্রেণি)
    dense_params_formula = d * k
    print(f"Weight matrix shape: {d} x {k}")
    print(f"Full fine-tuning (every entry trainable): {dense_params_formula:,} parameters\n")

    print(f"{'rank r':>8}{'formula: r*(d+k)':>20}{'actual LoRALinear':>20}{'% of dense':>14}")
    for r in [1, 4, 16, 64]:
        formula_count = r * (d + k)
        # Formula-টিকে একটি বাস্তবে instantiate করা module-এর parameter count-এর সাথে মিলিয়ে দেখা।
        layer = LoRALinear(d, k, r=r, bias=False)
        actual_trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
        pct = 100 * actual_trainable / dense_params_formula
        match = "OK" if formula_count == actual_trainable else "MISMATCH"
        print(f"{r:>8}{formula_count:>20,}{actual_trainable:>20,}{pct:>13.3f}%   [{match}]")

    print("\n-> Even at r=64, LoRA trains scarcely 3% as many parameters as full fine-tuning")
    print("   of this single matrix -- and real models apply LoRA to only a handful of")
    print("   projection matrices per layer (commonly just the attention Q and V")
    print("   projections), so the reduction across an entire model is even larger than")
    print("   this single-matrix comparison shows.")


parameter_count_demo()

## 4. QLoRA: frozen base-এর 4-bit block quantization-এর simulation

`quantize_4bit_blockwise` একটি সরলীকৃত QLoRA-ধাঁচের quantizer: প্রতি block-এ absmax scale factor আর একটি uniform 4-bit code (আসল NF4 non-uniform level ব্যবহার করে, কিন্তু mechanism একই)। Demo-টি একটি 4096 x 4096 Gaussian weight matrix-কে quantize করে fp32/fp16/4-bit storage-এর bytes তুলনা করে এবং relative reconstruction error মাপে — দেখায় কেন frozen base-কে 4 bit-এ রাখা যায় যখন ছোট A/B matrix-গুলো full precision-এ থাকে।

In [ ]:
# ---------------------------------------------------------------------------
# 4. QLoRA: frozen base-এর 4-bit block quantization simulate করা
# ---------------------------------------------------------------------------

def quantize_4bit_blockwise(weight, block_size=64):
    """QLoRA-ধাঁচের 4-bit blockwise quantization-এর একটি সরলীকৃত simulation
    (আসল NF4 non-uniform, empirically-fitted quantization level ব্যবহার করে যা
    pretrained weights-এর distribution-এর সাথে মেলানো; এটি প্রতি block-এ একটি
    uniform 4-bit grid ব্যবহার করে, যা সহজতর কিন্তু একই mechanism দেখায়:
    প্রতি-block scale factor + একটি shared low-bit code)।

    ফেরত দেয় (dequantized_weight, bytes_used, original_bytes_fp32)।
    """
    flat = weight.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)

    # প্রতি-block absmax scaling -- বাস্তব blockwise quantization হুবহু এটাই করে।
    absmax = blocks.abs().max(dim=1, keepdim=True).values.clamp_min(1e-8)
    LEVELS = 7  # এখানে ব্যবহৃত 4-bit signed range: integer -7..7 (15 level, 4 bit-এ এঁটে যায়)
    codes = torch.round(blocks / absmax * LEVELS).clamp(-LEVELS, LEVELS)
    dequantized_blocks = codes / LEVELS * absmax

    dequantized = dequantized_blocks.flatten()[:n].view_as(weight)

    num_blocks = blocks.shape[0]
    bytes_codes = n * 0.5           # 4 bits = প্রতি weight-এ 0.5 bytes
    bytes_scales = num_blocks * 4   # প্রতি block-এ একটি fp32 scale constant
    bytes_used = bytes_codes + bytes_scales
    bytes_fp32 = n * 4
    return dequantized, bytes_used, bytes_fp32


def qlora_demo():
    print("\n" + "=" * 78)
    print("4. QLoRA: 4-BIT BLOCKWISE QUANTIZATION OF THE FROZEN BASE (SIMULATED)")
    print("=" * 78)

    torch.manual_seed(1)
    # Pretrained weights মোটামুটি Gaussian-distributed -- NF4-এর non-uniform
    # quantization level-গুলো ঠিক এই বৈশিষ্ট্যকে ঘিরেই design করা।
    weight = torch.randn(4096, 4096)

    dequantized, bytes_4bit, bytes_fp32 = quantize_4bit_blockwise(weight, block_size=64)
    abs_error = (weight - dequantized).abs()
    rel_error = abs_error.mean().item() / weight.abs().mean().item()

    bytes_fp16 = weight.numel() * 2

    print(f"Weight matrix: {tuple(weight.shape)} = {weight.numel():,} values\n")
    print(f"{'storage format':<28}{'bytes':>14}{'vs fp32':>12}")
    print(f"{'fp32 (4 bytes/value)':<28}{bytes_fp32:>14,.0f}{'1.0x':>12}")
    print(f"{'fp16 (2 bytes/value)':<28}{bytes_fp16:>14,.0f}{bytes_fp32 / bytes_fp16:>11.1f}x")
    print(f"{'4-bit blockwise (simulated)':<28}{bytes_4bit:>14,.0f}{bytes_fp32 / bytes_4bit:>11.1f}x")

    print(f"\nMean absolute weight value:        {weight.abs().mean().item():.4f}")
    print(f"Mean absolute quantization error:  {abs_error.mean().item():.4f}")
    print(f"Relative error:                    {rel_error * 100:.2f}%")

    print(f"\n-> Storing the frozen base at 4 bits instead of fp32 shrinks its memory by")
    print(f"   {bytes_fp32 / bytes_4bit:.1f}x, for a {rel_error * 100:.1f}% relative reconstruction error -- a small enough")
    print("   error that the base model's forward pass stays useful, while its memory")
    print("   footprint drops enough to fit much larger models on a single GPU. This is")
    print("   exactly QLoRA's trick: quantize the FROZEN base (which never needs exact")
    print("   gradients), keep the small trainable A/B LoRA matrices in full precision")
    print("   (they're what's actually learning), and dequantize each block back to a")
    print("   working precision on the fly for each forward/backward pass.")


qlora_demo()

## সবগুলো demo একসাথে

`main()` function-টি ওপরের সব demo (`lora_layer_demo`, `parameter_count_demo`, `qlora_demo`) এক সাথে, ক্রমানুসারে চালায়। ওপরের cell-গুলোতে প্রতিটি demo ইতিমধ্যেই চলেছে, তাই নিচের call-টি comment করে রাখা হয়েছে — সবগুলো আবার একসাথে চালাতে চাইলে uncomment করুন।

In [ ]:
def main():
    lora_layer_demo()
    parameter_count_demo()
    qlora_demo()


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন